# RISC-V Memory Visualizer

## 1. Project Overview

An interactive educational visualizer connecting RISC-V instruction execution to
pipeline stages, memory addressing, row/column selection, SRAM storage and timing
events. Modular Python models perform simulation; Matplotlib views and Jupyter
widgets expose their state without executing it again.

**New here? Jump to [the final interactive demo](#final-demo).** Its code cell runs
by itself in a clean kernel. For the full learning path, run all cells in order.
Install with `python -m pip install -e ".[notebook,test]"` from the repository root,
then choose that environment's Python kernel. No local absolute paths are required.

This is a behavioral/logical educational model, not SPICE or a transistor-level
electrical, analog timing or physically accurate semiconductor simulator.
Waveforms show logical states, simulation cycles and event order, not voltages,
currents or physical delay. Only lw/sw/add/sub/addi are implemented.

The default SRAM has 16 **single-bit** cells at four-byte-spaced addresses 0 through
60. Abstract 8T lacks a source-verified transistor topology and does not imply
physical read/write ports or performance superiority. See the README and source
audit for assumptions. The first example introduces instructions and registers.


In [ ]:
from riscv_memory_visualizer.cpu.instruction import Instruction
from riscv_memory_visualizer.cpu.registers import RegisterFile
from riscv_memory_visualizer.simulation.state import SimulationState

In [ ]:
registers = RegisterFile()

registers.write(2, 0x100)

instruction = Instruction(
    opcode="lw",
    rd=5,
    rs1=2,
    immediate=8,
    original_text="lw x5, 8(x2)"
)

state = SimulationState()

print("Instruction:", instruction)
print("x2 =", hex(registers.read(2)))
print("Initial phase:", state.phase)

In [ ]:
registers.write(0,500)

print("x0 =", registers.read(0))

## 2. Instructions and Parsing

The parser converts assembly text into a structured Instruction object. It decodes fields without executing instructions.

Supported instructions are `lw`, `sw`, `add`, `sub`, and `addi`. Opcodes and `x0` through `x31` register names are case-insensitive. Commas are required; harmless whitespace is accepted. Signed decimal immediates are supported without an encoding-width restriction. ABI aliases and hexadecimal/binary immediates are not supported.

`original_text` preserves the input's case and internal whitespace after surrounding whitespace is stripped. Invalid syntax raises a descriptive `ValueError`. A store uses `rs2` for its data register and has no `rd`.

In [ ]:
from riscv_memory_visualizer.cpu.parser import InstructionParser

parser = InstructionParser()

In [ ]:
examples = [
    'lw x5, 8(x2)',
    'sw x6, 12(x3)',
    'add x3, x1, x2',
    'sub x8, x5, x4',
    'addi x4, x4, 1',
]

for text in examples:
    decoded = parser.parse(text)
    print('Instruction:', decoded.original_text)
    print('Opcode:', decoded.opcode)
    for label, register in [('RD', decoded.rd), ('RS1', decoded.rs1), ('RS2', decoded.rs2)]:
        print(f'{label}:', f'x{register}' if register is not None else '-')
    print('Immediate:', decoded.immediate if decoded.immediate is not None else '-')
    print()

## 3. 3-Stage RISC-V Pipeline

Each step first retires the previous MEM/WB instruction, then reads registers and executes the previous ID/EX instruction into MEM/WB, moves IF into ID/EX, and fetches the next instruction. Printed stages show occupancy after the step. The first instruction enters IF in cycle 1, ID/EX in cycle 2, MEM/WB in cycle 3, and retires in cycle 4.

Writeback happens before operand reads within a step. With this simplified timing, the adjacent dependent instructions below see the preceding results without forwarding, stalls, or hazard detection. This is an educational behavioral model. PC starts at 0 and advances by 4 only on a successful fetch; three final steps drain the pipeline.

DataMemory stores whole Python integers at byte addresses, with unwritten locations reading as zero. It models no SRAM, alignment checks, byte lanes, or 32-bit wrapping. A replacement backend needs only read(address) and write(address, value).

load_program() resets the CPU; initialize registers afterward if needed. reset() clears registers, cycle/PC, and pipeline while retaining the loaded program and memory. get_state() returns a detached SimulationState; its operation and effective_address describe retirement in the last active cycle. pipeline.snapshot() reports stage text separately, avoiding duplicate pipeline state. Stepping a drained CPU does nothing.

In [ ]:
from riscv_memory_visualizer.cpu.parser import InstructionParser
from riscv_memory_visualizer.cpu.processor import Processor

parser = InstructionParser()
program = [
    parser.parse('addi x2, x0, 256'),
    parser.parse('lw x5, 8(x2)'),
    parser.parse('add x6, x5, x2'),
    parser.parse('sw x6, 12(x2)'),
]
processor = Processor()
processor.load_program(program)
processor.memory.write(0x108, 42)

In [ ]:
while not processor.finished:
    cpu_state = processor.step()
    print('Cycle:', cpu_state.cpu_cycle)
    print('PC:', cpu_state.pc)
    for stage, text in processor.pipeline.snapshot().items():
        print(f'{stage}:', text if text is not None else '-')
    print('Retired:', cpu_state.instruction_text or '-')
    if cpu_state.operation is not None:
        print('Memory:', cpu_state.operation, hex(cpu_state.effective_address))
    print({f'x{i}': processor.registers.read(i) for i in (0, 2, 5, 6)})
    print()

assert processor.registers.read(5) == 42
assert processor.registers.read(6) == 298
assert processor.memory.read(0x10C) == 298
assert processor.registers.read(0) == 0
print('Stored value at 0x10c:', processor.memory.read(0x10C))

## 4. 6T SRAM Cell

This standalone behavioral cell models one stored bit in the cross-coupled storage nodes Q and Q_bar. Q_bar is always 1-Q. WL (word line) conceptually enables the access transistors connecting storage to BL and BL_bar. All five signals are binary logical integers, not analog voltages.

The default initial value is 0, WL is inactive, and both bit lines start prepared at 1. With WL=1, read() reports the stored bit without changing Q/Q_bar; its completed logical bit-line outcome is BL=Q and BL_bar=Q_bar. Preparation and read resolution are abstracted into one operation, with no transient timing or electrical simulation.

write(value) drives complementary bit lines and updates storage only with WL=1. Both operations leave WL unchanged. With WL=0, either access returns success=False and data=None without changing any signal. Lowering WL retains the previous bit-line values. reset() restores the constructor's initial bit, WL=0, and BL=BL_bar=1.

Results expose operation, success, data, an immutable state snapshot, and an explanation. Inputs must be integer 0 or 1; booleans, floats, and strings are rejected. The cell remains independent of Processor and DataMemory for future use in an SRAMArray.

In [ ]:
from riscv_memory_visualizer.memory.sram6t import SRAMCell6T

def show_cell(cell):
    snapshot = cell.get_state()
    for label, value in [('Q', snapshot.q), ('Q_bar', snapshot.q_bar),
                         ('WL', snapshot.word_line), ('BL', snapshot.bit_line),
                         ('BL_bar', snapshot.bit_line_bar)]:
        print(f'{label:7}: {value}')

cell = SRAMCell6T(initial_value=0)
print('6T SRAM Cell')
show_cell(cell)
print('Activating word line...')
cell.set_word_line(1)
print('WL:', cell.word_line)
read_result = cell.read()
print('Read result:', read_result.data, read_result.message)
assert read_result.success and read_result.data == 0
show_cell(cell)
print('Writing 1...')
write_result = cell.write(1)
print('Write success:', write_result.success, write_result.message)
show_cell(cell)
read_result = cell.read()
print('Read again:', read_result.data)
assert read_result.success and read_result.data == 1
assert cell.q == 1 and cell.q_bar == 0
print('Deactivating WL and attempting write(0)...')
cell.set_word_line(0)
blocked_result = cell.write(0)
print('Write success:', blocked_result.success, blocked_result.message)
assert not blocked_result.success and cell.q == 1
show_cell(cell)

## 5. SRAM Array

SRAMArray creates an independent SRAMCell6T at every (row, column). RowDecoder generates one-hot outputs for any positive row count; addresses outside 0 through rows-1 are rejected, including unused encoded addresses for non-power-of-two sizes.

Selecting a row sets WL=1 on every cell in that row and WL=0 in all other rows. Column selection only records which column to access. read(row, column) and write(row, column, value) select the location and call only that cell's existing operation, returning SRAMAccessResult with success, data, signal state, and explanation.

Selection and the selected row's WL remain active after an access for inspection. Other cells retain their storage and bit-line values. This logical model does not simulate shared electrical bit lines. reset() resets the existing cells and clears row/column selection.

get_state() returns immutable dimensions, selection, word lines, and stored-bit tuples. get_cell() returns a live cell for inspection/testing; use array methods for mutation to preserve selection semantics. Inputs are validated before state changes. Processor and DataMemory remain independent of this array.

In [ ]:
from riscv_memory_visualizer.memory.array import SRAMArray

array = SRAMArray(rows=4, columns=4, initial_value=0)
written = array.write(2, 3, 1)
assert written.success and written.data == 1
array_state = array.get_state()
print(f'SRAM Array: {array_state.rows} x {array_state.columns}')
print('Selected row:', array_state.selected_row)
print('Selected column:', array_state.selected_column)
print('Decoder:', array.row_decoder.get_state())
print('Word lines:')
for row, wl in enumerate(array_state.word_lines):
    print(f'WL{row} = {wl}')
print('Array contents:')
for row_values in array_state.cells:
    print(*row_values)
result = array.read(2, 3)
print('Read (2, 3):', result.data)
print(result.message)
assert result.success and result.data == 1
assert array.get_cell(2, 3).q == 1 and array.get_cell(2, 3).q_bar == 0
assert array.get_state().word_lines == (0, 0, 1, 0)
assert array.get_state().cells == array_state.cells
assert all(array.get_cell(r, c).q == int((r, c) == (2, 3))
           for r in range(4) for c in range(4))

## 6. Memory Events

MemoryEventEngine coordinates existing array operations and records immutable MemoryEvent observations in EventTimeline. read() returns data and write() returns None, matching the existing MemoryInterface; inspect engine.timeline.events() for the trace. The CPU is not connected in this phase. WRITEBACK is reserved for future integration and is not emitted.

Default byte-address mapping: index = address // 4; row = index // columns; column = index % columns. A 4 x 4 array accepts addresses 0, 4, ..., 60. Each location remains a single-bit cell (0 or 1), a placeholder at word-spaced addresses, not a 32-bit SRAM word. For example, address 36 (0x24) maps to (2, 1); 0x108 is outside this array.

Every event in an operation retains the supplied cycle (or None if unknown). Event order is insertion order, not CPU cycles; timelines never sort events. Instruction text is optional context and is not executed. Selection and WL remain active after SRAM access. Snapshots are immutable, and clearing the timeline leaves hardware unchanged.

Behavioral DataMemory is also supported with explicit capacity_words. It accepts integer data and emits request/response/completion events without inventing SRAM selection. Invalid inputs leave history and hardware unchanged. Backend exceptions propagate with only the observed event prefix retained, without a completion event.

In [ ]:
from riscv_memory_visualizer.memory.array import SRAMArray
from riscv_memory_visualizer.simulation.controller import MemoryEventEngine

event_array = SRAMArray(rows=4, columns=4)
engine = MemoryEventEngine(event_array)
engine.write(0x08, 1, cycle=5, instruction='sw x6, 8(x0)')
loaded_bit = engine.read(0x08, cycle=6, instruction='lw x5, 8(x0)')
assert loaded_bit == 1
assert event_array.get_cell(0, 2).q == 1
assert event_array.get_cell(0, 2).q_bar == 0
print('Memory Event Timeline')
for order, event in enumerate(engine.timeline.events(), 1):
    print(f'{order}. {event.event_type.name} | cycle={event.cycle}')
    print(f'   Address: {event.address:#04x}; row={event.row}; column={event.column}; value={event.value}')
    print('  ', event.description)
    if event.cell_state is not None:
        print('   Signals:', event.cell_state)
print('Returned data:', loaded_bit)

## 7. 6T Cell Visualization

Q / Q_bar store complementary bits in cross-coupled inverters. WL controls both access transistors; BL / BL_bar carry logical data. The six labeled boxes identify four inverter devices and two access devices. M1-M6 are local schematic labels. Supplies are symbolic; colors show logical state, not current or voltage.

The view only inspects state. It reuses one Figure and its artists. Explicit display calls preserve before/after notebook outputs. Event rendering uses historical snapshots; events without snapshots are rejected instead of borrowing current cell values. The lecture PDF was not available locally; this schematic follows the conceptual description supplied in the phase prompt.


In [ ]:
from IPython import get_ipython
from IPython.display import display
if get_ipython() is not None:
    get_ipython().run_line_magic('matplotlib', 'inline')
from riscv_memory_visualizer.memory.sram6t import SRAMCell6T
from riscv_memory_visualizer.visualization.cell_view import SRAMCell6TView
from riscv_memory_visualizer.memory.array import SRAMArray
from riscv_memory_visualizer.simulation.controller import MemoryEventEngine
from riscv_memory_visualizer.simulation.events import MemoryEventType

cell = SRAMCell6T(initial_value=0)
view = SRAMCell6TView()
print('Before write')
display(view.draw(cell))
cell.set_word_line(1)
cell.write(1)
print('After write: Q=1, Q_bar=0')
display(view.update(cell))

# Recorded write and read from the existing event engine.
visual_array = SRAMArray(1, 1)
visual_engine = MemoryEventEngine(visual_array)
visual_engine.write(0, 1)
visual_engine.read(0)
for event in visual_engine.timeline.events():
    if event.event_type in (MemoryEventType.SRAM_WRITE, MemoryEventType.SRAM_READ):
        display(view.render_event(event))


## 8. SRAM Array Visualization

Memory is organized as a 2-D array of cells. The row decoder selects a word line, enabling a row; column selection identifies the cell accessed through its bit-line pair. Blue shading indicates the active row or selected column. An orange border and the state panel identify their selected intersection. Values come directly from the live array.

The view reuses its figure and artists. Event rendering requires matching live selection and cell state; old events are rejected if they no longer match. It does not replay writes or reconstruct historical array contents. No electrical behavior is simulated.


In [ ]:
from IPython import get_ipython
from IPython.display import display
if get_ipython() is not None:
    get_ipython().run_line_magic('matplotlib', 'inline')
from riscv_memory_visualizer.memory.array import SRAMArray
from riscv_memory_visualizer.visualization.array_view import SRAMArrayView
from riscv_memory_visualizer.simulation.controller import MemoryEventEngine
from riscv_memory_visualizer.simulation.events import MemoryEventType

array = SRAMArray(rows=4, columns=4, initial_value=0)
array_view = SRAMArrayView()
display(array_view.draw(array))
array.write(2, 3, 1)
display(array_view.update(array))
assert array.read(2, 3).data == 1
display(array_view.update(array))

array_engine = MemoryEventEngine(array)
address = (2 * array.columns + 3) * 4
array_engine.write(address, 1, cycle=7)
write_event = next(e for e in array_engine.timeline.events() if e.event_type == MemoryEventType.SRAM_WRITE)
display(array_view.render_event(array, write_event))
array_engine.read(address, cycle=8)
read_event = next(e for e in array_engine.timeline.events() if e.event_type == MemoryEventType.SRAM_READ)
display(array_view.render_event(array, read_event))


## 9. Pipeline Visualization

IF fetches the next instruction. ID/EX decodes, reads registers, and performs ALU/address calculation. MEM/WB performs memory access when required and writes results back. The view shows actual stage occupancy after each step; ACTIVE means occupied, so several stages can be active together.

Only the notebook calls step(). The view observes state and existing latched results. In this model an instruction enters MEM/WB before retiring on the next step. Its memory indicator therefore describes pending access, not a newly executed request. The SRAM connection is conceptual; the processor continues to use behavioral DataMemory.

The reusable view also accepts processor.pipeline.snapshot() through render(snapshot, cycle=..., pc=...). Omitted cycle and PC remain unknown; no ALU values are reconstructed from text.


In [ ]:
from IPython import get_ipython
from IPython.display import display
if get_ipython() is not None:
    get_ipython().run_line_magic('matplotlib', 'inline')
from riscv_memory_visualizer.cpu.parser import InstructionParser
from riscv_memory_visualizer.cpu.processor import Processor
from riscv_memory_visualizer.visualization.pipeline_view import PipelineView

parser = InstructionParser()
processor = Processor()
processor.load_program([parser.parse(text) for text in [
    'addi x2, x0, 256', 'lw x5, 8(x2)', 'add x6, x5, x2'
]])
processor.memory.write(0x108, 42)
pipeline_view = PipelineView()
display(pipeline_view.draw(processor))
while not processor.finished:
    processor.step()
    display(pipeline_view.update(processor))
assert processor.registers.read(5) == 42
assert processor.registers.read(6) == 298


## 10. Integrated Application

The dashboard connects the existing processor to real SRAM accesses through
MemoryEventEngine. Its reusable views show pipeline occupancy, live array
selection, the selected cell, and digital timing. The chronological timeline records the
same accesses that return data to the CPU.

Run the next cell, click **Load Program**, then **Load Demo Data** to explicitly
write address 8 = 1. Press **Step** repeatedly or **Run**; **Pause** preserves
state, and Speed adjusts the interval. With the default three instructions,
x5 and x6 become 1 after six cycles. A single `lw x5, 8(x2)` with x2 initially
zero enters IF, ID/EX, and MEM/WB on steps 1-3; the actual SRAM read and register
writeback occur on step 4.

The 4 x 4 array is 16 logical single-bit cells, not full RISC-V words. Addresses
are 0, 4, ..., 60; stores accept only 0 or 1. Load and Reset clear SRAM and
history, so explicitly reload demo data afterward. Reset retains the loaded
program. Blank lines and # comments are accepted; parse errors preserve the
previous simulation. Setup events have no CPU cycle. Timeline selection is
reserved for a future enhancement; the diagrams always show live state.

Run uses the notebook event loop and yields between cycles. Re-executing the
cell closes the previous app to avoid leaving an automatic run active.

In [ ]:
from riscv_memory_visualizer.ui.app import RiscVMemoryVisualizerApp

if 'app' in globals():
    app.close()
app = RiscVMemoryVisualizerApp()
app.display()

## 11. 6T vs Abstract 8T

An SRAM cell is not defined only by the bit it stores. Its transistor-level
organization determines how storage, read, and write paths are implemented.

**Source limitation:** the available project has no lecture PDF or 8T
connectivity specification. Source material does not specify enough 8T
transistor connectivity to implement a source-faithful 8T schematic.

The existing 6T project schematic shows six devices, complementary storage,
and WL-controlled access through BL/BL_bar. The 8T brief specifies a transistor
count but does not establish device roles, wiring, or separate read/write
signals. Its view below is an **abstract logical block**, not eight invented
devices. Q/Q_bar complementarity, software selection, non-destructive logical
read and bit replacement on write are implementation assumptions. No analog
or performance comparison is made. Physical 8T structures remain unknown.

The next cell uses the same application and CPU to run `lw x5,8(x2)` in both
configurations. Each switch clears execution, SRAM and history, retains the
loaded program, and pauses automatic execution. Demo data is loaded explicitly.
The 8T case demonstrates the equivalent logical result, not physical simulation.
Use the selector in the dashboard afterward to explore either configuration.

In [ ]:
from IPython.display import display
from riscv_memory_visualizer.ui.app import RiscVMemoryVisualizerApp

if 'app' not in globals():
    app = RiscVMemoryVisualizerApp()
    app.display()
app.pause()
original_cpu = app.processor
for architecture in ('6T', '8T'):
    app.architecture_selector.value = architecture
    assert app.load_program('lw x5,8(x2)')
    app.load_demo_data()  # Explicit initialization: address 8 = 1.
    for _ in range(4):
        assert app.step()
    assert app.processor is original_cpu
    assert app.processor.registers.read(5) == 1
    assert (app.array.selected_row, app.array.selected_column) == (0, 2)
    print(architecture, 'behavioral result: x5 = 1; selected cell (0, 2)')
    display(app.cell_view.figure)


### Side-by-side model comparison

These are independent example cells with the same logical value. The left
view reuses the existing 6T schematic; the right explicitly withholds an 8T
transistor schematic. Separate `read()` and `write()` software methods do not
establish separate physical ports. See `docs/sram_architectures.md` for the
source audit and assumptions.

In [ ]:
from riscv_memory_visualizer.memory.sram6t import SRAMCell6T
from riscv_memory_visualizer.memory.sram8t import SRAMCell8T
from riscv_memory_visualizer.visualization.cell_comparison_view import SRAMCellComparisonView

comparison_6t = SRAMCell6T()
comparison_8t = SRAMCell8T()
comparison_6t.set_word_line(1)
comparison_8t.set_selected(1)  # Software gate, not a physical word line.
comparison_6t.write(1)
comparison_8t.write(1)
assert comparison_6t.read().data == comparison_8t.read().data == 1
comparison_view = SRAMCellComparisonView()
display(comparison_view.draw(comparison_6t, comparison_8t))


## 12. Digital Timing / Waveforms

### What is a waveform?
A waveform shows how a signal changes as simulation time advances.

### What is being shown here?
The visualizer records logical hardware states and simulation events. The upper
panel shows cycle-boundary observations; the lower panel shows event order within
one cycle. Event positions are not extra CPU cycles or physical durations.

### What is not being modeled?
This waveform represents logical simulation state and event timing. It is
not an analog SPICE waveform. Transistor voltages, currents, capacitances,
and physical delays are not modeled.

The example uses a 4 x 4 6T array initialized to a known bit, 1. We observe
cell (0, 2), read address 8 at supplied cycle 1, then write 0 at supplied cycle 2.
These are explicit demonstration cycle labels; the event engine does not advance
a CPU. WL remains selected after access, as defined by the existing array model.
The recorder and view only observe the simulation.

In [ ]:
from IPython.display import display
from riscv_memory_visualizer.memory.array import SRAMArray
from riscv_memory_visualizer.simulation.controller import MemoryEventEngine
from riscv_memory_visualizer.simulation.events import MemoryEventType as E
from riscv_memory_visualizer.simulation.signals import SimulationTraceRecorder
from riscv_memory_visualizer.visualization.waveform_view import WaveformView

timing_array = SRAMArray(4, 4, initial_value=1)
timing_engine = MemoryEventEngine(timing_array)
timing_recorder = SimulationTraceRecorder(timing_engine.timeline)
timing_view = WaveformView()
timing_recorder.record_cycle(0, timing_array, cell=(0, 2))
assert timing_engine.read(8, cycle=1) == 1
timing_recorder.record_cycle(1, timing_array, cell=(0, 2))
read_events = timing_engine.timeline.get_cycle(1)
assert [e.event_type for e in read_events] == [
    E.MEMORY_REQUEST, E.ADDRESS_DECODE, E.ROW_SELECT, E.COLUMN_SELECT,
    E.WORD_LINE_ACTIVE, E.SRAM_READ, E.MEMORY_RESPONSE, E.OPERATION_COMPLETE,
]
assert all(e.cycle == 1 for e in read_events)
print('Read event order:', ' -> '.join(e.event_type.name for e in read_events))
for name in ('CYCLE', 'WL', 'Q', 'Q_bar', 'READ', 'WRITE'):
    print(name, timing_recorder.cycle_traces[name].values())
display(timing_view.draw(timing_recorder.snapshot(), event_cycle=1))

### Write and observe the stored bit changing
The same recorder and figure now include a write. Q changes from 1 to 0 and
Q_bar from 0 to 1. The event panel shows the before/after snapshots within
cycle 2; READ and WRITE mark access occurrences, not control voltages.

In [ ]:
timing_engine.write(8, 0, cycle=2)
timing_recorder.record_cycle(2, timing_array, cell=(0, 2))
assert timing_recorder.cycle_traces['Q'].values() == (1, 1, 0)
assert timing_recorder.cycle_traces['Q_bar'].values() == (0, 0, 1)
assert timing_recorder.cycle_traces['READ'].values() == (0, 1, 0)
assert timing_recorder.cycle_traces['WRITE'].values() == (0, 0, 1)
for name in ('CYCLE', 'WL', 'Q', 'Q_bar', 'READ', 'WRITE'):
    print(name, timing_recorder.cycle_traces[name].values())
display(timing_view.update(timing_recorder.snapshot(), event_cycle=2))

### Integrated application waveform
The existing dashboard records actual CPU cycles, PC, IF, ID_EX and MEM_WB,
alongside the memory events generated by those instructions. Step and Run extend
history; Pause preserves it. Reset, Load Program and architecture changes clear
history. The default view shows the latest 12 cycles.

The example below reuses the dashboard from Phase 10 and displays its waveform
here as well. In the dashboard, select **8T (abstract)** to inspect Q/Q_bar and
software selection. Unsupported WL/BL/BL_bar are N/A; no physical 8T paths are
invented. After Reset or Load Program, explicitly reload demo data if needed.

In [ ]:
from riscv_memory_visualizer.ui.app import RiscVMemoryVisualizerApp

if 'app' not in globals():
    app = RiscVMemoryVisualizerApp()
    app.display()
app.pause()
app.architecture_selector.value = '6T'
assert app.load_program('lw x5,8(x2)')
app.load_demo_data()
for _ in range(4):
    assert app.step()
assert app.processor.registers.read(5) == 1
assert app.trace_recorder.cycle_traces['CYCLE'].values() == (0, 1, 2, 3, 4)
paused_trace = app.trace_recorder.snapshot()
app.pause()
assert app.trace_recorder.snapshot() == paused_trace
display(app.waveform_view.figure)

<a id="final-demo"></a>

## 13. Interactive Demo

# Final Interactive Demonstration

**This is the main application. Run the next cell independently in a clean kernel.**

1. Click **Load Program** to load the default three instructions.
2. Click **Load Demo Data** to explicitly set address 8 to 1.
3. Use **Step** six times, or **Run**, to finish with x5=1 and x6=1.
4. Observe the pipeline, registers, selected cell (0, 2), timeline and waveform.
5. **Pause** preserves history; **Speed** controls the wait between steps.
6. **Reset** retains the program but clears SRAM and history; reload demo data.
7. Select **8T (abstract)** to compare the software model; switching resets
   execution and SRAM, so reload demo data again. Unsupported physical signals
   remain N/A.

The cell closes an earlier app before replacing it, so rerunning does not leave
an automatic run active. No hidden initialization or previous cells are needed.


In [1]:
from riscv_memory_visualizer.ui.app import RiscVMemoryVisualizerApp

if 'app' in globals():
    app.close()
app = RiscVMemoryVisualizerApp()
app.display()